In [10]:
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.io import fits

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from dustmaps.edenhofer2023 import Edenhofer2023Query
import dustmaps.edenhofer2023

import sys
sys.path.append("/home/leone/Documents/Uni/Bachelor/Project/")
import FuncDef as fd
from importlib import reload
reload(fd)

import itertools

from jinja2 import Template


savefolder = "../PlotlyTesting/savedplots/website/"

In [2]:
data = fd.load_data("/home/leone/Documents/Uni/Bachelor/Project/cloud-data.csv", sep=";")
colors = ["red", "orange", "blue", "green", "cyan", "turquoise", "gray", "yellow", "palegreen", "peru", "gold", "olive", "teal", "navy", "fuchsia", "magenta", "crimson", "violet", "olivedrab", "maroon", "cornflowerblue", "chocolate", "darkorchid"]

a = pd.read_csv("/home/leone/Documents/Uni/Bachelor/Project/results.csv")
a.set_index("i", inplace=True)
#delete column Unnamed:0
a.drop(a.columns[0], axis=1, inplace=True)
for i in a.columns:
    a[i] = a[i].apply(lambda x: x.replace(" ", ""))
    a[i] = a[i].apply(lambda x: x.replace(",", "."))
    a[i] = a[i].astype(float)
#create data_e which is data + mean column from a
data_e = data.copy()
data_e["mean"] = a["mean"]
FACTOR = 1653

In [3]:
clouds = {}
volumes = list()
for i in data_e.index:
    #get index for color
    try:
        index = int(i)
    except ValueError:
        index = int(i[:-1])
    cl = fd.Cloud(*data.loc[i])
    vol = cl.getVolume(colors[index-1], i, nx=20, ny=20, nz=20)
    vol["hovertemplate"] = "Mass: " + str(data_e["mean"].loc[i]/1e3) + " 10<sup>3</sup>M<sub>☉</sub><br>"
    clouds[i] = (vol)

Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this might take a couple of seconds)...
Optimizing map for querying (this 

In [4]:
sun = go.Scatter3d(
    x=[0],
    y=[0],
    z=[0],
    mode="markers",
    marker=dict(
        size=2,
        color="yellow"
    ),
    name = "Sun",
    showlegend=True,
)

In [5]:
xrange = [-1250, 1250]
yrange = [-1250, 1250]
zrange = [-625, 625]
xf = np.arange(xrange[0], xrange[1], 25)
yf = np.arange(xrange[0], xrange[1], 25)
zf = np.arange(xrange[0], xrange[1], 25)

xf, yf, zf = np.meshgrid(xf, yf, zf)

dust_dist = fd.query_region(xf, yf, zf)

Optimizing map for querying (this might take a couple of seconds)...


In [6]:
dust_vol = go.Volume(
    x=xf.flatten(),
    y=yf.flatten(),
    z=zf.flatten(),
    value=dust_dist,
    isomin=2e-4,
    isomax=1e-2,
    opacity=0.15,
    opacityscale=[[0, 0.4], [1,0.75]],
    surface_count=10,
    colorscale="Greys",
    showscale=False,
    visible="legendonly",
    name = "E+ [E]",
    showlegend=True,
)

xscale = (xrange[1] - xrange[0]) / (zrange[1] - zrange[0])
yscale = (yrange[1] - yrange[0]) / (zrange[1] - zrange[0])

layout = go.Layout(
    template="plotly_white",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    scene=dict(
        aspectmode="manual",
        aspectratio=dict(x=xscale, y=yscale, z=1),
        xaxis=dict(range=xrange, title=dict(text="X [pc]", font_size=30)),
        yaxis=dict(range=yrange, title=dict(text="Y [pc]", font_size=30)),
        zaxis=dict(range=zrange, title=dict(text="Z [pc]", font_size=30)),
        #camera=dict(
        #    up=dict(x=0, y=1, z=0),
        #    eye=dict(x=0, y=0, z=3.7),
        #),
    ),
    width= 1000,
    height=1000,
    font=dict(
        size=14,
        color="black",
    ),
)

In [8]:
fig = go.Figure(data=[sun], layout=layout)
#fig["data"][0]["showlegend"] = False
fig.add_trace(dust_vol)
#fig["data"][1]["isomin"]=1e-4
#fig["layout"]["template"] = "plotly_dark"

for i in clouds.keys():
    fig.add_trace(clouds[i])
    fig["data"][-1]["showlegend"] = False

In [11]:
output_path = r"clouds.md"
template_path = r"figure_empty_clouds.md"


plotly_jinja_data = {"fig": fig.to_html(include_plotlyjs = "cdn", full_html=False)}

with open(output_path, "w", encoding="utf-8") as output_file:
    with open(template_path) as template_file:
        j2_template = Template(template_file.read())
        output_file.write(j2_template.render(plotly_jinja_data))